# 04 — Main evaluation: Qwen2.5-Omni-7B (cross-model robustness)

Runs every stage in `DEFAULT_STAGE_ORDER` over the full available
manifest. Saves one JSON per stage to `qwen2_5_omni_7b/`.

**Runtime:** ~10-14 hr on Colab A100 for 600 samples × 7 stages. Needs ≥40GB VRAM.
**Resume-safe:** if Colab times out, rerun the cell — qa_ids already done
are skipped.


In [ ]:
# ─── Colab bootstrap ────────────────────────────────────────
# Mount Drive (caches model weights + videos across sessions),
# clone the repo, install dependencies, and configure the cache dirs.
import os, sys, subprocess, pathlib

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/avut"
    REPO_DIR = "/content/omnimodel-research"
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    if not os.path.exists(REPO_DIR):
        subprocess.run([
            "git", "clone",
            "https://github.com/samadasyed/omnimodel-research.git",
            REPO_DIR,
        ], check=True)
else:
    DRIVE_ROOT = os.path.expanduser("~/avut")
    REPO_DIR = str(pathlib.Path.cwd())
    os.makedirs(DRIVE_ROOT, exist_ok=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Persistent storage layout (data + model caches under DRIVE_ROOT)
DATA_DIR        = os.path.join(DRIVE_ROOT, "data")
VIDEO_DIR       = os.path.join(DATA_DIR, "videos")
AUDIO_DIR       = os.path.join(DATA_DIR, "audio")
SILENT_DIR      = os.path.join(DATA_DIR, "silent")
TRANSCRIPT_DIR  = os.path.join(DATA_DIR, "transcripts")
ANNOTATION_DIR  = os.path.join(DATA_DIR, "annotations")
RESULTS_DIR     = os.path.join(DRIVE_ROOT, "results")
RAW_PRED_DIR    = os.path.join(RESULTS_DIR, "raw_predictions")
METRICS_DIR     = os.path.join(RESULTS_DIR, "metrics")
HF_CACHE        = os.path.join(DRIVE_ROOT, ".cache", "hf")
WHISPER_CACHE   = os.path.join(DRIVE_ROOT, ".cache", "whisper")

for d in [VIDEO_DIR, AUDIO_DIR, SILENT_DIR, TRANSCRIPT_DIR,
          ANNOTATION_DIR, RAW_PRED_DIR, METRICS_DIR, HF_CACHE, WHISPER_CACHE]:
    os.makedirs(d, exist_ok=True)

# Redirect HF cache to Drive so we don't redownload weights each session
os.environ["HF_HOME"] = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE
os.environ["HF_DATASETS_CACHE"] = HF_CACHE

print(f"Repo:       {REPO_DIR}")
print(f"Drive root: {DRIVE_ROOT}")
print(f"HF cache:   {HF_CACHE}")


In [ ]:
# ─── Install model dependencies ──────────────────────────
# transformers (Gemma-3n needs a recent build), accelerate, qwen-omni-utils
# (only loaded when running 05_main_eval_qwen), imageio for frame sampling,
# soundfile + librosa for audio I/O, openai-whisper for ASR.
%pip install -q -U "transformers>=4.55.0" accelerate>=0.30     huggingface-hub>=0.24 imageio[ffmpeg] soundfile librosa scipy
%pip install -q openai-whisper
# Qwen-Omni preview branch is only needed for notebook 05; install when running it.


In [ ]:
import json, time, gc
import torch
from src import data_utils, stages
from src.stages import STAGE_REGISTRY, DEFAULT_STAGE_ORDER

manifest_path = os.path.join(DATA_DIR, "sample_manifest_available.json")
with open(manifest_path) as f:
    samples = json.load(f)
print(f"Manifest: {len(samples)} samples")

PRED_DIR = os.path.join(RAW_PRED_DIR, "qwen2_5_omni_7b")
os.makedirs(PRED_DIR, exist_ok=True)

# Mismatched-transcript pairings (built in notebook 02)
mm_path = os.path.join(DATA_DIR, "mismatched_pairs.json")
mismatched_pairs = {}
if os.path.exists(mm_path):
    with open(mm_path) as f:
        mismatched_pairs = {int(k): v for k, v in json.load(f).items()}
print(f"Mismatched pairings: {len(mismatched_pairs)}")


In [ ]:
# Stages to run — change to a subset if you want to skip some
STAGES_TO_RUN = "S1_text_only,S2_audio_only,S3_visual_only,S4_full_av,S5_transcript_injected,S6_attribution,S7_mismatched_transcript".split(",")
print("Will run:", STAGES_TO_RUN)


In [ ]:
def get_paths(s, mismatched_pairs):
    vid = s["video_id"]
    paths = {
        "video":  os.path.join(VIDEO_DIR, f"{vid}.mp4"),
        "silent": os.path.join(SILENT_DIR, f"{vid}_silent.mp4"),
        "audio":  os.path.join(AUDIO_DIR, f"{vid}.wav"),
        "transcript": os.path.join(TRANSCRIPT_DIR, f"{vid}.txt"),
    }
    donor = mismatched_pairs.get(s["qa_id"])
    if donor:
        paths["mismatched_transcript"] = os.path.join(TRANSCRIPT_DIR, f"{donor}.txt")
        paths["donor_video_id"] = donor
    return paths


def stage_path(stage_name):
    return os.path.join(PRED_DIR, f"{stage_name}.json")


def load_existing(stage_name):
    p = stage_path(stage_name)
    if not os.path.exists(p):
        return []
    with open(p) as f:
        return json.load(f)


def save_rows(stage_name, rows):
    with open(stage_path(stage_name), "w") as f:
        json.dump(rows, f, indent=2)


def already_done_ids(stage_name):
    return {r["qa_id"] for r in load_existing(stage_name)
            if r.get("predicted_answer") is not None}


In [ ]:
# Qwen-Omni preview branch needs a specific transformers ref:
%pip install -q "git+https://github.com/huggingface/transformers@v4.51.3-Qwen2.5-Omni-preview"
%pip install -q "qwen-omni-utils[decord]>=0.0.4"

from src.model_utils import QwenOmniWrapper

def load_omni_model():
    return QwenOmniWrapper(model_name="Qwen/Qwen2.5-Omni-7B")


In [ ]:
def run_stage(stage_name, samples, model, s4_lookup=None):
    spec = STAGE_REGISTRY[stage_name]
    fn = spec["fn"]
    done = already_done_ids(stage_name)
    rows = load_existing(stage_name)
    todo = [s for s in samples if s["qa_id"] not in done]
    print(f"\n[{stage_name}] {len(done)} done, {len(todo)} to go")

    t0 = time.time()
    for i, s in enumerate(todo, 1):
        paths = get_paths(s, mismatched_pairs)
        try:
            if stage_name == "S6_attribution":
                s4_row = s4_lookup.get(s["qa_id"])
                if s4_row is None:
                    continue
                row = fn(model, s, paths, s4_row)
            else:
                row = fn(model, s, paths)
        except Exception as e:
            row = {
                "qa_id": s["qa_id"], "video_id": s["video_id"],
                "task_type": s["task_type"], "task_code": s.get("task_code"),
                "stage": stage_name,
                "error": f"{type(e).__name__}: {str(e)[:200]}",
                "predicted_answer": None, "confidence": None,
            }
        rows.append(row)
        if i % 25 == 0:
            save_rows(stage_name, rows)
            elapsed = time.time() - t0
            rate = elapsed / i
            print(f"  [{i}/{len(todo)}] {rate:.1f}s/sample, "
                  f"ETA {rate * (len(todo) - i):.0f}s")
    save_rows(stage_name, rows)
    print(f"  saved {len(rows)} rows to {stage_path(stage_name)}")
    return rows


# Phase A: text-only stage (small Qwen2.5-1.5B)
text_stages = [s for s in STAGES_TO_RUN if STAGE_REGISTRY[s]["model"] == "text"]
if text_stages:
    from src.model_utils import TextOnlyModelWrapper
    print("Loading Qwen2.5-1.5B-Instruct (text-only baseline)...")
    text_model = TextOnlyModelWrapper()
    for stage in text_stages:
        run_stage(stage, samples, text_model)
    del text_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Phase B: omnimodal stages
omni_stages = [s for s in STAGES_TO_RUN if STAGE_REGISTRY[s]["model"] == "omni"]
if omni_stages:
    omni_model = load_omni_model()
    s4_lookup = {}
    for stage in omni_stages:
        if stage == "S6_attribution" and not s4_lookup:
            s4_rows = load_existing("S4_full_av")
            s4_lookup = {r["qa_id"]: r for r in s4_rows}
            if not s4_lookup:
                print("  S4_full_av not run yet; skipping S6_attribution.")
                continue
        run_stage(stage, samples, omni_model, s4_lookup=s4_lookup)
    del omni_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nDone. Predictions in:", PRED_DIR)


Next: `06_analysis.ipynb` to compute metrics on `qwen2_5_omni_7b/`.